<a href="https://colab.research.google.com/github/arashshams/Insurance-Policy-RAG/blob/new_dev/notebooks/01_document_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Insurance Policy RAG - Notebook 01: Document Ingestion

This is the **first stage** of the Insurance Policy RAG pipeline.

**Goal:** turn a single insurance policy PDF into clean, page-tagged, token-aware text chunks, and save them for the embedding stage.

**Steps in this notebook:**
1. Mount Drive and set up the project folder structure
2. Locate the policy PDF (single source of truth)
3. Extract text page-by-page, preserving page numbers for citations
4. Split the text into overlapping, token-aware chunks with page metadata
5. Save the chunks to `artifacts/chunks.json` for **Notebook 02 (Embeddings & Indexing)**

> Note: the policy PDF is not committed to the repo for privacy. Place your own PDF in `data/documents/` before running the PDF-dependent cells.

In [16]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Insurance-Policy-RAG')
for sub in ['notebooks', 'data/documents', 'artifacts', 'chroma']:
    (PROJECT_ROOT / sub).mkdir(parents=True, exist_ok=True)
print('Project root ready at:', PROJECT_ROOT)
print('Subfolders:', [p.name for p in PROJECT_ROOT.iterdir()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root ready at: /content/drive/MyDrive/Insurance-Policy-RAG
Subfolders: ['notebooks', 'data', 'artifacts', 'chroma']


In [ ]:
# Install dependencies for this notebook (safe to re-run)
# On a fresh Colab runtime these packages are not pre-installed.
!pip install -q pypdf tiktoken langchain-text-splitters > /dev/null 2>&1
print("Dependencies installed.")

In [ ]:
# Standard library
import os
import json
import time
from types import SimpleNamespace

# Numerical utilities
import numpy as np
from tqdm import tqdm

# Tokenization (for token-aware chunking)
import tiktoken

# PDF parsing
from pypdf import PdfReader

# Recursive, token-aware text splitter (standalone package; no full langchain needed)
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries imported successfully.")

## 2. Configure the Policy Document Path

This notebook uses a **single insurance policy PDF** as the source of truth.

Place your policy PDF inside the project's `data/documents/` folder on Drive:

`/content/drive/MyDrive/Insurance-Policy-RAG/data/documents/`

The cell below automatically detects the first PDF found in that folder. If no PDF is present yet, it raises a clear message so you know to add one.

In [ ]:
# Locate the policy PDF inside the project's data/documents folder
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"

pdf_paths = sorted(DOCUMENTS_DIR.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(
        f"No PDF found in {DOCUMENTS_DIR}.\n"
        "Upload your insurance policy PDF into that folder and re-run this cell."
    )

if len(pdf_paths) > 1:
    names = ", ".join(p.name for p in pdf_paths)
    print(f"Warning: {len(pdf_paths)} PDFs found ({names}).")
    print("Using the first alphabetically as the single source of truth.")

# Single source of truth: use the first PDF found
PDF_PATH = pdf_paths[0]
print("Using policy document:", PDF_PATH.name)

## 3. Extract Text (Page by Page)

We extract text one page at a time and keep the page number as metadata.
Preserving page numbers is what makes **citations** possible later in the Q&A notebook.

We also run a quick sanity check: if most pages come back empty, the PDF is likely a
scanned image (no text layer) and would need OCR before it can be used.

In [ ]:
def _scramble_ratio(text):
    """Fraction of whitespace-delimited tokens that are a single character.

    Cleanly extracted prose sits near ~0.05-0.15. Pages rendered from a
    broken/decorative embedded font (e.g. an image-heavy cover page) come
    back letter-spaced and score very high (> 0.5).
    """
    toks = text.split()
    if not toks:
        return 0.0
    return sum(1 for t in toks if len(t) == 1) / len(toks)


# Pages whose scramble ratio exceeds this are treated as unreadable and skipped.
# Tune upward for legitimately table-heavy or acronym-heavy documents.
SCRAMBLE_THRESHOLD = 0.5


def extract_pages(pdf_path):
    """Extract text page-by-page, preserving 1-based page numbers.

    A per-page quality gate drops pages that come back empty (no text layer)
    or scrambled/letter-spaced (an image-heavy cover page with a defective
    embedded font), so we never feed garbage into embeddings downstream.

    Returns a list of dicts: [{"page": int, "text": str}, ...]
    """
    try:
        reader = PdfReader(str(pdf_path))
    except Exception as e:
        raise RuntimeError(
            f"Could not open PDF ({pdf_path.name}). "
            f"It may be corrupted or unsupported: {e}"
        )
    if getattr(reader, "is_encrypted", False):
        raise RuntimeError(
            f"{pdf_path.name} appears to be encrypted. "
            "Remove the password protection and re-run this cell."
        )

    pages = []
    skipped = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = "\n".join(line.strip() for line in text.splitlines())

        if not text.strip():
            skipped.append((i, "empty"))
            continue

        ratio = _scramble_ratio(text)
        if ratio > SCRAMBLE_THRESHOLD:
            skipped.append((i, round(ratio, 2)))
            continue

        pages.append({"page": i, "text": text})

    if skipped:
        details = ", ".join(f"p{n} ({r})" for n, r in skipped)
        print(f"Quality gate: skipped {len(skipped)} unreadable page(s): {details}")

    if not pages:
        raise RuntimeError(
            f"No extractable text found in {pdf_path.name}. "
            "The PDF is likely scanned images with no text layer and needs "
            "OCR before it can be ingested."
        )

    return pages


pages = extract_pages(PDF_PATH)
print(f"Extracted {len(pages)} usable pages from {PDF_PATH.name}")

In [21]:
# Extraction sanity check
non_empty = [p for p in pages if p["text"].strip()]
print(f"Pages with extractable text: {len(non_empty)} / {len(pages)}")

if len(non_empty) == 0:
    print("WARNING: No text extracted. This PDF may be a scanned image and require OCR.")
else:
    sample = non_empty[0]
    print(f"\n--- Page {sample['page']} preview (first 500 chars) ---")
    print(sample["text"][:500])

Pages with extractable text: 45 / 45

--- Page 2 preview (first 500 chars) ---
Loblaw Companies Limited 1

Loblaw Companies Limited


Plan Contract Number:   G0100180,  G0100181

Group Policy Number:   G0100179

Plan BH  - Extra Foods and West Real Canadian Superstore - UFCW 832 - active full time


Employee Name: ______________________________

Certificate Numbe r: ____________________________









Welcome to Your Group Benefit Program

Plan Contract Effective Date:
G0100180  - October 1, 2015
G0100181  - December 1, 2015

Group Policy Effective Date:      October 1, 


## 4. Chunk the Text (Token-Aware, with Metadata)

LLMs cannot process an entire long policy at once, so we split the text into smaller,
overlapping **chunks**. We use a token-aware splitter so each chunk stays within a
predictable token budget, and we keep the page number on every chunk so answers can
cite their source.

- `CHUNK_SIZE`: target tokens per chunk (larger = more context, fewer chunks)
- `CHUNK_OVERLAP`: tokens shared between neighbouring chunks (preserves context across boundaries)

In [ ]:
# Token-aware splitter using the cl100k_base encoding for accurate token counts.
# We keep the proven RecursiveCharacterTextSplitter (same chunking behaviour) but
# wrap its output in lightweight SimpleNamespace objects instead of LangChain
# Document objects, so the notebook no longer depends on langchain / langchain-core.
CHUNK_SIZE = 800      # target tokens per chunk
CHUNK_OVERLAP = 128   # tokens shared between neighbouring chunks

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Build one set of chunks per page, carrying the page number as metadata.
# split_text returns plain strings; we attach metadata ourselves.
docs = []
for p in pages:
    if not p["text"].strip():
        continue
    for chunk_text in text_splitter.split_text(p["text"]):
        docs.append(
            SimpleNamespace(page_content=chunk_text, metadata={"page": p["page"]})
        )

print(f"Created {len(docs)} chunks from {len(pages)} pages")
if docs:
    print("\n--- First chunk preview ---")
    print("page:", docs[0].metadata.get("page"))
    print(docs[0].page_content[:400])

## 5. Save Chunks for the Next Stage

Finally, we persist the chunks (text + page metadata) to `artifacts/chunks.json` in the
project folder. Notebook **02 (embeddings & indexing)** loads this file directly, so the
embedding step doesn't have to re-parse and re-chunk the PDF every time.

In [23]:
# Persist chunks (text + metadata) so notebook 02 can load them directly
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CHUNKS_PATH = ARTIFACTS_DIR / "chunks.json"

records = [
    {
        "id": f"doc_{i}",
        "text": d.page_content,
        "page": int(d.metadata.get("page", -1)),
        "source": PDF_PATH.name,
    }
    for i, d in enumerate(docs)
]

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} chunks to: {CHUNKS_PATH}")

Saved 45 chunks to: /content/drive/MyDrive/Insurance-Policy-RAG/artifacts/chunks.json
